# Étape 1 : Extraction du texte et des images

3.1. Définir le chemin du PDF

In [2]:
from pathlib import Path
import pymupdf

PDF_PATH = Path("./pdfs/attention is all you need.pdf")

OUTPUT_DIR = Path("resultats_pdf")
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"

if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"Fichier introuvable : {PDF_PATH.resolve()}"
    )

TEXT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF : {PDF_PATH.resolve()}")
print(f"Résultats : {OUTPUT_DIR.resolve()}")

PDF : C:\Users\Lenovo\Downloads\RAG-Multi-Modal-Agentique\pdfs\attention is all you need.pdf
Résultats : C:\Users\Lenovo\Downloads\RAG-Multi-Modal-Agentique\resultats_pdf


3.2. Extraire le texte et les images

In [3]:
textes = []
nombre_images = 0
pages_sans_texte = []

with pymupdf.open(PDF_PATH) as document:

    nombre_pages = len(document)

    for numero_page, page in enumerate(document, start=1):

        # 1. Extraction du texte
        texte = page.get_text("text", sort=True)

        # 2. Sauvegarde du texte de chaque page
        fichier_texte = TEXT_DIR / f"page_{numero_page:03d}.txt"

        fichier_texte.write_text(
            texte,
            encoding="utf-8"
        )

        textes.append(
            f"===== PAGE {numero_page} =====\n\n{texte}"
        )

        # 3. Extraction des images
        images_vues = set()

        for image_info in page.get_images(full=True):

            xref = image_info[0]

            if xref in images_vues:
                continue

            images_vues.add(xref)

            image = document.extract_image(xref)

            if not image:
                continue

            extension = image["ext"]

            fichier_image = (
                IMAGE_DIR /
                f"page_{numero_page:03d}_image_{xref}.{extension}"
            )

            fichier_image.write_bytes(image["image"])
            nombre_images += 1

# 4. Sauvegarder le texte intégral
(OUTPUT_DIR / "texte_complet.txt").write_text(
    "\n\n".join(textes),
    encoding="utf-8"
)

print("Pages traitées :", nombre_pages)
print("Images enregistrées :", nombre_images)

Pages traitées : 11
Images enregistrées : 3


# Étape 2 : Vectorisation dans un espace latent commun (Embeddings)

Cette étape charge **OpenCLIP**, encode conjointement le texte et les images avec **ViT-B-32**, puis normalise les vecteurs pour permettre des comparaisons fiables par similarité cosinus.

In [2]:
from pathlib import Path
import torch
import open_clip
from PIL import Image

TEXT_DIR = Path("resultats_pdf/textes")
IMAGE_DIR = Path("resultats_pdf/images")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED,
)
model = model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

print("Appareil :", DEVICE)
print("Modèle :", MODEL_NAME, "préentraîné sur", PRETRAINED)

c:\Users\Lenovo\Downloads\RAG-Multi-Modal-Agentique\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

c:\Users\Lenovo\Downloads\RAG-Multi-Modal-Agentique\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--laion--CLIP-ViT-B-32-laion2B-s34B-b79K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Appareil : cpu
Modèle : ViT-B-32 préentraîné sur laion2b_s34b_b79k


## Encodage et normalisation des embeddings

Les vecteurs texte et image sont produits par les deux encodeurs de CLIP. Chaque vecteur est ensuite divisé par sa norme afin d’obtenir une norme égale à 1. Cette normalisation est essentielle avant le calcul de similarités cosinus.

In [3]:
def normaliser_embeddings(vecteurs):
    """Normaliser des embeddings 1D ou 2D pour obtenir une norme L2 de 1."""
    return vecteurs / vecteurs.norm(dim=-1, keepdim=True).clamp_min(1e-12)

@torch.inference_mode()
def encoder_textes(textes, taille_lot=32):
    """Encoder une liste de textes dans l’espace latent commun CLIP."""
    textes_valides = [texte.strip() for texte in textes if texte and texte.strip()]
    if not textes_valides:
        return []

    lots = []
    for debut in range(0, len(textes_valides), taille_lot):
        tokens = tokenizer(textes_valides[debut:debut + taille_lot]).to(DEVICE)
        vecteurs = model.encode_text(tokens).float()
        vecteurs = normaliser_embeddings(vecteurs)
        lots.append(vecteurs.cpu())

    return torch.cat(lots, dim=0).tolist()

@torch.inference_mode()
def encoder_images(chemins, taille_lot=32):
    """Encoder une liste d’images dans l’espace latent commun CLIP."""
    vecteurs_lots = []

    for debut in range(0, len(chemins), taille_lot):
        tenseurs = []
        for chemin in chemins[debut:debut + taille_lot]:
            with Image.open(chemin) as image:
                tenseurs.append(preprocess(image.convert("RGB")))

        if not tenseurs:
            continue

        tenseurs = torch.stack(tenseurs).to(DEVICE)
        vecteurs = model.encode_image(tenseurs).float()
        vecteurs = normaliser_embeddings(vecteurs)
        vecteurs_lots.append(vecteurs.cpu())

    if not vecteurs_lots:
        return []

    return torch.cat(vecteurs_lots, dim=0).tolist()

In [4]:
# Charger les textes et les images extraits à l’étape précédente
chemins_textes = sorted(TEXT_DIR.glob("*.txt"))
extensions_images = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
chemins_images = [
    chemin for chemin in sorted(IMAGE_DIR.iterdir())
    if chemin.is_file() and chemin.suffix.lower() in extensions_images
]

textes_pages = [chemin.read_text(encoding="utf-8") for chemin in chemins_textes]
embeddings_textes = encoder_textes(textes_pages)
embeddings_images = encoder_images(chemins_images)

print("Pages vectorisées :", len(embeddings_textes))
print("Images vectorisées :", len(embeddings_images))

for type_embedding, embeddings in (("texte", embeddings_textes), ("image", embeddings_images)):
    if embeddings:
        tenseur = torch.tensor(embeddings, dtype=torch.float32)
        normes = tenseur.norm(dim=-1)
        print(
            f"{type_embedding.capitalize()} : dimension={tenseur.shape[1]}, "
            f"norme moyenne={normes.mean().item():.6f}, "
            f"écart maximal à 1={torch.abs(normes - 1).max().item():.2e}"
        )

Pages vectorisées : 11
Images vectorisées : 3
Texte : dimension=512, norme moyenne=1.000000, écart maximal à 1=5.96e-08
Image : dimension=512, norme moyenne=1.000000, écart maximal à 1=5.96e-08


# Étape 3 : Indexation et stockage dans une base vectorielle

Cette étape initialise une base **ChromaDB persistante**, calcule une empreinte SHA-256 du PDF et enregistre les embeddings texte et image avec leurs métadonnées.

## Initialisation de ChromaDB

La collection utilise la distance cosinus. Avant une réindexation, les anciennes entrées associées au même nom de PDF sont supprimées afin d’éviter les doublons ou les contenus obsolètes.

In [5]:
import hashlib
import re

import chromadb

PDF_PATH = Path("pdfs/attention is all you need.pdf")
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF introuvable : {PDF_PATH.resolve()}")

document_id = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()

client = chromadb.PersistentClient(path="chroma_pdf")
collection = client.get_or_create_collection(
    name="pdf_clip_vit_b32",
    embedding_function=None,
    metadata={"hnsw:space": "cosine"},
)

collection.delete(where={"pdf": PDF_PATH.name})

print("Identifiant du document :", document_id[:24])
print("Anciennes entrées du même PDF supprimées")

Identifiant du document : d87d482d5ae7960e2e43d7dd
Anciennes entrées du même PDF supprimées


In [6]:
def extraire_numero_page(chemin):
    """Extraire le numéro de page depuis un nom du type page_001.txt."""
    correspondance = re.search(r"page_(\d+)", Path(chemin).stem)
    return int(correspondance.group(1)) if correspondance else -1

ids_textes = []
embeddings_textes_indexes = []
documents_textes = []
metadatas_textes = []

for fichier, embedding in zip(chemins_textes, embeddings_textes):
    page = extraire_numero_page(fichier)
    texte = fichier.read_text(encoding="utf-8")

    if not texte.strip():
        continue

    ids_textes.append(f"{document_id[:24]}:texte:{page}")
    embeddings_textes_indexes.append(embedding)
    documents_textes.append(texte)
    metadatas_textes.append({
        "type": "texte",
        "document_id": document_id[:24],
        "pdf": PDF_PATH.name,
        "page": page,
        "source": str(fichier.resolve()),
    })

if ids_textes:
    collection.add(
        ids=ids_textes,
        embeddings=embeddings_textes_indexes,
        documents=documents_textes,
        metadatas=metadatas_textes,
    )

ids_images = []
documents_images = []
metadatas_images = []

for fichier, embedding in zip(chemins_images, embeddings_images):
    page = extraire_numero_page(fichier)
    ids_images.append(f"{document_id[:24]}:image:{fichier.name}")
    documents_images.append(f"Image extraite du PDF : {fichier.name}")
    metadatas_images.append({
        "type": "image",
        "document_id": document_id[:24],
        "pdf": PDF_PATH.name,
        "page": page,
        "source": str(fichier.resolve()),
    })

if ids_images:
    collection.add(
        ids=ids_images,
        embeddings=embeddings_images,
        documents=documents_images,
        metadatas=metadatas_images,
    )

print("Fragments texte indexés :", len(ids_textes))
print("Images indexées :", len(ids_images))
print("Total dans ChromaDB :", collection.count(), "éléments")

Fragments texte indexés : 11
Images indexées : 3
Total dans ChromaDB : 14 éléments


# Étape 4 : Recherche multimodale (Retrieval)

Cette étape utilise la similarité cosinus pour retrouver les $k$ éléments les plus proches d’une requête textuelle ou d’une requête image.

In [7]:
def rechercher(texte=None, image=None, type_resultat=None, k=5):
    """Rechercher les k éléments multimodaux les plus proches de la requête."""
    if (texte is None) == (image is None):
        raise ValueError("Fournissez soit un texte, soit une image, pas les deux.")
    if k <= 0:
        raise ValueError("k doit être strictement positif.")
    if type_resultat not in (None, "texte", "image"):
        raise ValueError("type_resultat doit être None, 'texte' ou 'image'.")

    if texte is not None:
        vecteur_requete = encoder_textes([texte])[0]
        requete_description = texte
    else:
        chemin_image = Path(image)
        if not chemin_image.is_file():
            raise FileNotFoundError(f"Image introuvable : {chemin_image.resolve()}")
        vecteur_requete = encoder_images([chemin_image])[0]
        requete_description = str(chemin_image)

    filtre = None if type_resultat is None else {"type": type_resultat}
    nombre_disponible = len(collection.get(where=filtre, include=[])["ids"])
    nombre_resultats = max(1, min(k, nombre_disponible))

    if nombre_disponible == 0:
        return {"requete": requete_description, "mode": "texte" if texte is not None else "image", "resultats": []}

    resultats = collection.query(
        query_embeddings=[vecteur_requete],
        n_results=nombre_resultats,
        where=filtre,
        include=["documents", "metadatas", "distances"],
    )

    elements = []
    for identifiant, document, metadata, distance in zip(
        resultats["ids"][0],
        resultats["documents"][0],
        resultats["metadatas"][0],
        resultats["distances"][0],
    ):
        elements.append({
            "id": identifiant,
            "document": document,
            "metadata": metadata,
            "distance_cosinus": float(distance),
            "similarite_cosinus": float(1.0 - distance),
        })

    return {
        "requete": requete_description,
        "mode": "texte" if texte is not None else "image",
        "resultats": elements,
    }

## Affichage des résultats

Cette fonction affiche les métadonnées et la similarité de chaque résultat. Les textes sont imprimés intégralement ou tronqués pour l’affichage, et les images sont ouvertes depuis leur chemin source.

In [9]:
import matplotlib.pyplot as plt

def afficher_resultats(recherche):
    """Afficher les k résultats renvoyés par rechercher."""
    resultats = recherche["resultats"]
    print("Requête", recherche["mode"] + " :", recherche["requete"])

    if not resultats:
        print("Aucun résultat disponible.")
        return

    apercus_images = []
    for rang, resultat in enumerate(resultats, start=1):
        metadata = resultat["metadata"]
        print(f"\nRésultat {rang}")
        print("  Similarité cosinus :", f"{resultat['similarite_cosinus']:.4f}")
        print("  Type :", metadata.get("type"))
        print("  Page :", metadata.get("page"))
        print("  Source :", metadata.get("source"))

        if metadata.get("type") == "texte":
            document = resultat["document"]
            print("  Texte :", document[:800] + ("..." if len(document) > 800 else ""))
        else:
            apercus_images.append((metadata["source"], rang))

    if apercus_images:
        nombre_colonnes = min(3, len(apercus_images))
        nombre_lignes = (len(apercus_images) + nombre_colonnes - 1) // nombre_colonnes
        figure, axes = plt.subplots(nombre_lignes, nombre_colonnes, figsize=(5 * nombre_colonnes, 4 * nombre_lignes))
        axes = [axes] if nombre_lignes * nombre_colonnes == 1 else axes.ravel()

        for axe, (chemin, rang) in zip(axes, apercus_images):
            with Image.open(chemin) as image:
                axe.imshow(image.convert("RGB"))
            axe.set_title(f"Résultat {rang}")
            axe.axis("off")

        for axe in axes[len(apercus_images):]:
            axe.axis("off")

        plt.tight_layout()
        plt.show()

### Exemples d’utilisation

```python
resultats = rechercher(texte="What is the Transformer architecture?", k=5)
afficher_resultats(resultats)

resultats = rechercher(
    texte="Attention mechanism",
    type_resultat="image",
    k=3,
)
afficher_resultats(resultats)

premiere_image = chemins_images[0] if chemins_images else None
if premiere_image:
    resultats = rechercher(image=premiere_image, k=5)
    afficher_resultats(resultats)
```

# Étape 5 : Transformation du Retriever en outil agentique

Le retriever est encapsulé dans un outil LangChain nommé `retriever_tool`. Le nom, la description, les types d’arguments et les métadonnées permettent à un agent IA de décider quand et comment l’utiliser.

In [10]:
import json
from typing import Literal

from langchain.tools import tool

@tool
def retriever_tool(
    question: str = "",
    image: str | None = None,
    type_resultat: Literal["tous", "texte", "image"] = "tous",
    k: int = 5,
) -> str:
    """Rechercher dans les textes et les images du PDF avec CLIP et ChromaDB.

    Utiliser cet outil lorsqu’une question concerne le contenu documentaire,
    les concepts, les chiffres, les méthodes, les résultats ou les figures.
    Fournir soit une question textuelle, soit le chemin d’une image. Ne pas fournir
    les deux à la fois.

    Paramètres :
    - question : requête textuelle en langue naturelle.
    - image : chemin local vers une image utilisée comme requête visuelle.
    - type_resultat : "tous", "texte" ou "image".
    - k : nombre maximal de résultats à retourner, entre 1 et 20.
    """
    if not 1 <= k <= 20:
        raise ValueError("k doit être compris entre 1 et 20.")

    question = question.strip()
    if bool(question) == bool(image):
        raise ValueError("Fournissez soit une question textuelle, soit une image.")

    filtre = None if type_resultat == "tous" else type_resultat
    recherche = rechercher(
        texte=question or None,
        image=image or None,
        type_resultat=filtre,
        k=k,
    )

    sortie = {
        "mode": recherche["mode"],
        "requete": recherche["requete"],
        "nombre_resultats": len(recherche["resultats"]),
        "resultats": [],
    }

    for resultat in recherche["resultats"]:
        sortie["resultats"].append({
            "id": resultat["id"],
            "similarite_cosinus": resultat["similarite_cosinus"],
            "metadonnees": resultat["metadata"],
            "contenu": resultat["document"][:1200],
        })

    return json.dumps(sortie, ensure_ascii=False)

retriever_tool.metadata = {
    "tool_type": "retriever_multimodal",
    "backend": "ChromaDB",
    "embedding_model": "ViT-B-32",
    "pretrained": "laion2b_s34b_b79k",
    "distance": "cosine",
    "document_id": document_id[:24],
    "collection": collection.name,
}

print("Outil créé :", retriever_tool.name)
print("Description :", retriever_tool.description)
print("Métadonnées :", retriever_tool.metadata)

Outil créé : retriever_tool
Description : Rechercher dans les textes et les images du PDF avec CLIP et ChromaDB.

    Utiliser cet outil lorsqu’une question concerne le contenu documentaire,
    les concepts, les chiffres, les méthodes, les résultats ou les figures.
    Fournir soit une question textuelle, soit le chemin d’une image. Ne pas fournir
    les deux à la fois.

    Paramètres :
    - question : requête textuelle en langue naturelle.
    - image : chemin local vers une image utilisée comme requête visuelle.
    - type_resultat : "tous", "texte" ou "image".
    - k : nombre maximal de résultats à retourner, entre 1 et 20.
Métadonnées : {'tool_type': 'retriever_multimodal', 'backend': 'ChromaDB', 'embedding_model': 'ViT-B-32', 'pretrained': 'laion2b_s34b_b79k', 'distance': 'cosine', 'document_id': 'd87d482d5ae7960e2e43d7dd', 'collection': 'pdf_clip_vit_b32'}


## Schéma exposé à l’agent

`@tool` transforme la fonction Python en outil LangChain. Le LLM reçoit notamment :

- le nom `retriever_tool` ;
- la description extraite de la docstring ;
- les paramètres `question`, `image`, `type_resultat` et `k` ;
- les types et valeurs autorisées pour `type_resultat` ;
- les métadonnées du modèle, de la base et du document.

# Étape 6 : Construction de l’agent IA avec LangGraph

Cette étape compose le modèle de raisonnement (**MIND**) avec l’outil de recherche multimodale (**BODY**). `create_agent` compile automatiquement un graphe LangGraph ReAct : le modèle raisonne, appelle l’outil, observe le résultat, puis reprend ou répond.

In [27]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, ToolMessage

load_dotenv(dotenv_path=Path(".env"), override=False)

if not os.environ.get("GROQ_API_KEY"):
    raise EnvironmentError(
        "GROQ_API_KEY est absent du fichier .env. "
        "Ajoutez une ligne GROQ_API_KEY=votre_clé dans le fichier .env."
    )

MODELE_AGENT = "openai/gpt-oss-20b"
modele_agent = ChatGroq(model=MODELE_AGENT, temperature=0)

INSTRUCTIONS_AGENT = """
Tu es un assistant documentaire multimodal pour le PDF indexé.
Tu disposes de l’outil retriever_tool, qui recherche dans les passages
textuels et les images avec CLIP et ChromaDB.

Règles :
1. Pour toute question portant sur le contenu du PDF, utilise retriever_tool.
2. Utilise type_resultat="texte" pour les explications, chiffres et passages.
3. Utilise type_resultat="image" pour les figures ou diagrammes.
4. Utilise type_resultat="tous" lorsqu’il faut comparer les deux modalités.
5. Formule des recherches précises et effectue plusieurs appels si nécessaire.
6. Cite les pages et les chemins sources fournis par l’outil.
7. Ne prétends pas avoir consulté une source absente des résultats.
8. Réponds en français, sauf demande explicite d’utiliser une autre langue.
"""

agent = create_agent(
    model=modele_agent,
    tools=[retriever_tool],
    system_prompt=INSTRUCTIONS_AGENT,
    name="agent_rag_multimodal",
)

print("Agent compilé : agent_rag_multimodal")
print("Cerveau (LLM Groq) :", MODELE_AGENT)
print("Corps (outil) :", retriever_tool.name)

Agent compilé : agent_rag_multimodal
Cerveau (LLM Groq) : openai/gpt-oss-20b
Corps (outil) : retriever_tool


## Boucle ReAct compilée

`create_agent` retourne un `CompiledStateGraph`. Le graphe décide, appelle l’outil, reçoit les observations et peut poursuivre plusieurs tours avant la réponse finale.

In [19]:
def interroger_agent(question, k=5, type_resultat="tous", afficher=True):
    """Invoquer l’agent ReAct avec une question en langue naturelle."""
    if not question or not question.strip():
        raise ValueError("La question ne peut pas être vide.")
    if type_resultat not in ("tous", "texte", "image"):
        raise ValueError("type_resultat doit être 'tous', 'texte' ou 'image'.")

    execution = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"{question.strip()}\n"
                        f"Type de sources souhaité : {type_resultat}. "
                        f"Nombre de résultats demandé : {k}."
                    ),
                }
            ]
        },
        config={"recursion_limit": 20},
    )

    appels_outils = []
    observations = []
    for message in execution["messages"]:
        if isinstance(message, AIMessage):
            for appel in message.tool_calls:
                appels_outils.append({"name": appel["name"], "args": appel["args"]})
        elif isinstance(message, ToolMessage):
            observations.append(message.content)

    reponse_finale = execution["messages"][-1].content
    if afficher:
        display(reponse_finale)

    return {
        "reponse": reponse_finale,
        "appels_outils": appels_outils,
        "observations": observations,
        "messages": execution["messages"],
    }

# Étape 7 : Tests et validation de l’agent RAG

Cette étape soumet des questions simples, hybrides et multi-étapes afin d’analyser la décision d’appeler l’outil, les paramètres choisis, les contextes observés, les tours ReAct et la réponse finale.

In [24]:
TESTS_AGENT = [
    {
        "nom": "Contribution principale",
        "question": "Quelle est la contribution principale de l’article Attention Is All You Need ?",
        "type_resultat": "texte",
        "k": 6,
    },
    {
        "nom": "Figure du Transformer",
        "question": "Retrouve les passages et figures décrivant l’architecture encodeur-décodeur du Transformer.",
        "type_resultat": "tous",
        "k": 8,
    },
    {
        "nom": "Recherche hybride multi-étapes",
        "question": "Compare l’attention multi-têtes et l’encodage positionnel. Recherche séparément leurs explications textuelles et les figures associées, puis fais une synthèse avec les pages sources.",
        "type_resultat": "tous",
        "k": 5,
    },
    {
        "nom": "Récupération visuelle",
        "question": "Identifie les figures liées à Scaled Dot-Product Attention et Multi-Head Attention, puis explique les opérations représentées.",
        "type_resultat": "image",
        "k": 4,
    },
]

In [25]:
def analyser_observations(observations):
    """Compter les résultats retournés par chaque observation JSON."""
    nombre_resultats = 0
    types_observes = set()

    for observation in observations:
        try:
            contenu = json.loads(observation)
        except (TypeError, json.JSONDecodeError):
            continue

        nombre_resultats += len(contenu.get("resultats", []))
        for resultat in contenu.get("resultats", []):
            type_resultat = resultat.get("metadonnees", {}).get("type")
            if type_resultat:
                types_observes.add(type_resultat)

    return nombre_resultats, sorted(types_observes)

def executer_suite_tests(tests=TESTS_AGENT):
    """Reconstruire l’agent avec le modèle disponible, puis exécuter les tests."""
    global MODELE_AGENT, modele_agent, agent

    MODELE_AGENT = "openai/gpt-oss-20b"
    modele_agent = ChatGroq(model=MODELE_AGENT, temperature=0)
    agent = create_agent(
        model=modele_agent,
        tools=[retriever_tool],
        system_prompt=INSTRUCTIONS_AGENT,
        name="agent_rag_multimodal",
    )
    print("Modèle utilisé pour les tests :", MODELE_AGENT)

    resultats = []
    for test in tests:
        try:
            sortie = interroger_agent(
                question=test["question"],
                k=test["k"],
                type_resultat=test["type_resultat"],
                afficher=False,
            )
            appels_retriever = [
                appel for appel in sortie["appels_outils"]
                if appel["name"] == "retriever_tool"
            ]
            nombre_resultats, types_observes = analyser_observations(sortie["observations"])

            resultats.append({
                "nom": test["nom"],
                "question": test["question"],
                "type_demande": test["type_resultat"],
                "outil_appele": bool(appels_retriever),
                "nombre_appels_retriever": len(appels_retriever),
                "nombre_resultats_observes": nombre_resultats,
                "types_observes": types_observes,
                "reponse_non_vide": bool(str(sortie["reponse"]).strip()),
                "appels": appels_retriever,
                "reponse": sortie["reponse"],
                "erreur": None,
            })
        except Exception as erreur:
            resultats.append({
                "nom": test["nom"],
                "question": test["question"],
                "type_demande": test["type_resultat"],
                "outil_appele": False,
                "nombre_appels_retriever": 0,
                "nombre_resultats_observes": 0,
                "types_observes": [],
                "reponse_non_vide": False,
                "appels": [],
                "reponse": "",
                "erreur": str(erreur),
            })

    return resultats

def afficher_bilan_tests(resultats):
    """Afficher un bilan lisible pour chaque requête de validation."""
    for resultat in resultats:
        print("=" * 72)
        print(resultat["nom"])
        print("Question :", resultat["question"])
        print("Outil appelé :", resultat["outil_appele"])
        print("Appels retriever :", resultat["nombre_appels_retriever"])
        print("Résultats observés :", resultat["nombre_resultats_observes"])
        print("Types observés :", ", ".join(resultat["types_observes"]) or "aucun")
        print("Réponse non vide :", resultat["reponse_non_vide"])

        for appel in resultat["appels"]:
            print("Arguments outil :", json.dumps(appel["args"], ensure_ascii=False))

        if resultat["erreur"]:
            print("Erreur :", resultat["erreur"])
        else:
            print("Réponse :")
            print(resultat["reponse"])

## Exécution de la validation

Exécuter la cellule suivante uniquement après avoir exécuté les étapes 1 à 6 et configuré `GROQ_API_KEY`. Elle envoie les quatre tests à l’agent et affiche les décisions ReAct.

In [26]:
resultats_tests_agent = executer_suite_tests()
afficher_bilan_tests(resultats_tests_agent)

Modèle utilisé pour les tests : openai/gpt-oss-20b
Contribution principale
Question : Quelle est la contribution principale de l’article Attention Is All You Need ?
Outil appelé : True
Appels retriever : 1
Résultats observés : 6
Types observés : texte
Réponse non vide : True
Arguments outil : {"k": 6, "question": "Quelle est la contribution principale de l’article Attention Is All You Need ?", "type_resultat": "texte"}
Réponse :
**Contribution principale de l’article *Attention Is All You Need***  

L’article introduit le **Transformer**, une architecture de réseau neuronal entièrement basée sur des mécanismes d’attention (attention multi‑tête et attention auto‑régressive), sans aucune couche récurrente ni convolutionnelle. Cette approche permet :

1. **D’éliminer la dépendance séquentielle** des modèles RNN/GRU/LSTM, ce qui rend l’entraînement **parallélisable** sur GPU/TPU.  
2. **D’obtenir des performances de traduction** (BLEU) supérieures à celles des meilleurs modèles existants (